# Module 04 · Unit 02
# Directed Graphs and DAGs

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Neural Network Architecture \[NN\] · Attack Graphs & Reachability \[AG\] |
| **Refresher** | Module 04 · Unit 01 — Graph Definitions |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define a **directed graph** (digraph) and distinguish it from an undirected graph
2. Compute **in-degree** and **out-degree** for every vertex
3. Define a **Directed Acyclic Graph (DAG)** and explain why acyclicity matters
4. Compute a **topological ordering** of a DAG and prove it exists iff the graph is acyclic
5. Explain why the forward pass of a neural network is a DAG traversal
6. Represent an agentic AI tool call sequence as a DAG and detect unsafe cycles


## 🔣 Symbol Primer

Adding direction to graphs introduces a small set of new notation.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $(u, v)$ | Directed edge / arc | "arc from $u$ to $v$" | An ordered connection: $u$ points to $v$ |
| $\deg^+(v)$ | Out-degree | "out-degree of $v$" | Number of edges leaving $v$ |
| $\deg^-(v)$ | In-degree | "in-degree of $v$" | Number of edges entering $v$ |
| $\text{DAG}$ | Directed Acyclic Graph | "dag" | A digraph with no directed cycles |
| $u \rightsquigarrow v$ | Reachability | "$u$ reaches $v$" | A directed path from $u$ to $v$ exists |
| $\text{topo}(G)$ | Topological order | "topological ordering of $G$" | A linear order on vertices respecting all edges |
| $\text{Sources}(G)$ | Source vertices | "sources of $G$" | Vertices with $\deg^-(v) = 0$ — no incoming edges |
| $\text{Sinks}(G)$ | Sink vertices | "sinks of $G$" | Vertices with $\deg^+(v) = 0$ — no outgoing edges |

> **The upgrade from Unit 01.** In an undirected graph, $\{u,v\}$ and $\{v,u\}$
> are the same edge. In a directed graph, $(u,v)$ and $(v,u)$ are *different* arcs
> — the first says $u$ can reach $v$; the second says $v$ can reach $u$.
> Direction is what makes dependency, causality, and computation expressible as graphs.

---


## 1 · Directed Graphs

A **directed graph** (digraph) $G = (V, E)$ uses ordered pairs for edges:

$$E \subseteq V \times V \setminus \{(v,v) \mid v \in V\}$$

*(no self-loops in a simple digraph)*

The arc $(u, v)$ goes *from* $u$ *to* $v$. The arc $(v, u)$ is a different arc
going the other direction. Both may exist, one may exist, or neither.

### In-Degree and Out-Degree

Each vertex now has two degree measures:

$$\deg^+(v) = |\{(v, u) \in E\}| \quad \text{(out-degree — arcs leaving } v\text{)}$$
$$\deg^-(v) = |\{(u, v) \in E\}| \quad \text{(in-degree — arcs entering } v\text{)}$$

The directed analogue of the Handshaking Lemma:

$$\sum_{v \in V} \deg^+(v) = \sum_{v \in V} \deg^-(v) = |E|$$

Every arc contributes exactly 1 to the out-degree sum and 1 to the in-degree sum.

### Security Examples of Directed Graphs

| System | Vertices | Arc $(u,v)$ meaning |
|---|---|---|
| **Dependency graph** | Packages / modules | $u$ requires $v$ |
| **Call graph** | Functions | $u$ calls $v$ |
| **Attack graph** | Systems / vulnerabilities | $u$ can be exploited to reach $v$ |
| **Data flow graph** | Variables / operations | Data flows from $u$ to $v$ |
| **Trust graph** | Principals | $u$ grants trust to $v$ |

Direction is the difference between *"A depends on B"* and *"B depends on A"*.
These are very different things — conflating them is a common source of both
logical errors and security misconfigurations.


## 2 · Directed Paths and Reachability

A **directed path** from $u$ to $v$ is a sequence $u = v_0, v_1, \ldots, v_k = v$
where $(v_i, v_{i+1}) \in E$ for all $i$ — each step follows an arc in the
forward direction.

**Reachability:** $u \rightsquigarrow v$ means a directed path from $u$ to $v$
exists. Reachability is *not* symmetric in a digraph: $u \rightsquigarrow v$
does not imply $v \rightsquigarrow u$.

### Reachability as a Security Property

$$\text{compromised}(u) \wedge u \rightsquigarrow v \;\Rightarrow\; \text{potentially compromised}(v)$$

*"If a host is compromised and can reach another host, the second is at risk."*

This is the formal foundation of **attack graph reachability** — the core
topic of Module 05. For now, note that reachability in a directed graph is
a $\exists$ claim:

$$u \rightsquigarrow v \;\equiv\; \exists \text{ directed path } p \text{ from } u \text{ to } v$$

And its negation (the security guarantee that isolation is effective):

$$\neg(u \rightsquigarrow v) \;\equiv\; \forall \text{ walks from } u,\; v \text{ is never reached}$$

### Strongly and Weakly Connected

A digraph is **strongly connected** if $u \rightsquigarrow v$ and $v \rightsquigarrow u$
for every pair $u, v$ — every vertex can reach every other.

A digraph is **weakly connected** if the *underlying undirected graph* (ignoring
directions) is connected.

**Security:** A strongly connected component (SCC) in an attack graph is a set of
systems where, if any one is compromised, all others are potentially reachable via
directed paths. An SCC is a cluster of mutually reachable systems — a natural
unit of blast radius analysis.


## 3 · Directed Acyclic Graphs (DAGs)

A **DAG** is a directed graph with no directed cycles — you can never return to
a starting vertex by following arcs forward.

**Formal definition:** $G = (V, E)$ is a DAG if there is no sequence
$v_0, v_1, \ldots, v_k = v_0$ (with $k \geq 1$) where each $(v_i, v_{i+1}) \in E$.

### Why Acyclicity Matters

**Computation:** If a computation graph has a cycle, execution would loop forever.
Every function that terminates corresponds to a DAG of operations.

**Causality:** In a causal model, causes precede effects. A cycle would mean
$A$ causes $B$ causes $A$ — a causal paradox. DAGs are the formal language of
causal models.

**Dependency resolution:** Package managers (pip, npm, cargo) resolve dependencies
by topologically ordering a DAG. A cycle means "A requires B requires A" —
an unresolvable circular dependency.

**Neural networks:** The forward pass of a feedforward neural network is a DAG
traversal — data flows from inputs through layers to outputs with no cycles.
Recurrent networks (RNNs) add cycles deliberately, but at the cost of requiring
careful handling of the temporal unrolling.

### Sources, Sinks, and the Structure of a DAG

- **Sources** ($\deg^- = 0$): no incoming arcs. In a neural network these are
  the *input neurons*. In a dependency graph, packages with no dependencies.
- **Sinks** ($\deg^+ = 0$): no outgoing arcs. In a neural network these are
  the *output neurons*. In a dependency graph, packages nothing depends on.

Every DAG has at least one source and at least one sink. This is a consequence
of acyclicity — if you follow arcs forward you must eventually terminate at a sink.


## 4 · Topological Ordering

A **topological ordering** (or topological sort) of a DAG $G = (V, E)$ is a
linear ordering of all vertices such that for every arc $(u, v) \in E$,
vertex $u$ appears before $v$ in the ordering.

$$\text{topo}(G) = [v_1, v_2, \ldots, v_n] \quad\text{such that}\quad (v_i, v_j) \in E \Rightarrow i < j$$

### Existence Theorem

$$G \text{ has a topological ordering} \;\Longleftrightarrow\; G \text{ is a DAG}$$

**Proof sketch ($\Rightarrow$):** If $G$ has a cycle $v_0 \to v_1 \to \cdots \to v_0$,
then the ordering would require $v_0$ before $v_1$ before $\cdots$ before $v_0$ —
a contradiction. So a topological ordering implies no cycles.

**Proof sketch ($\Leftarrow$):** Kahn's algorithm: repeatedly remove sources
(vertices with in-degree 0) from the graph, appending them to the ordering.
If $G$ is a DAG, this process terminates with all vertices ordered.
If $G$ has a cycle, the process stalls — the cyclic vertices never reach in-degree 0.

### Security and Operational Relevance

| Context | Topological order means |
|---|---|
| Package installation | Install dependencies before dependents |
| Build systems | Compile files in dependency order |
| Database migrations | Run migrations in correct sequence |
| Neural network forward pass | Compute layers in order — no layer needs a future layer's output |
| Agent tool calls | Execute tools in the order their outputs feed into subsequent tools |

**Cycle = security risk in agent systems.** If an agent's tool call graph has a
cycle — Tool A's output feeds Tool B, whose output feeds Tool A — the agent could
loop indefinitely. We detect this in the Python section below.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[NN\] \[AG\]

| DAG concept | \[NN\] Neural Networks | \[AG\] Attack Graphs |
|---|---|---|
| **Vertex** | Neuron / operation / layer | Host, service, or vulnerability |
| **Arc $(u,v)$** | Data flows from $u$ to $v$ | $u$ can be exploited to reach $v$ |
| **Source** | Input neuron (receives data) | Entry point (attacker's foothold) |
| **Sink** | Output neuron (produces prediction) | Target (attacker's goal) |
| **Topological order** | Forward pass execution order | Lateral movement sequence |
| **Cycle** | Infinite loop — invalid feedforward net | Persistent access loop |
| **Reachability** | Can gradient flow? | Can attacker move from entry to target? |
| **Removing a vertex** | Model surgery / ablation | Taking down a compromised host |

**The deepest connection.** The forward pass of a neural network and the analysis
of an attack graph are the *same mathematical operation*: a DAG traversal that
propagates values (activations or exploit outcomes) from sources to sinks following
directed arcs. The security engineer and the ML engineer are doing the same thing
at different layers of the same system.

---


## Python: Visualization & Verification

**1 · Digraph Construction and Degree Analysis** — build a software dependency
digraph, compute in-degrees and out-degrees, identify sources and sinks, and
flag circular dependencies.

**2 · DAG Detection and Topological Sort** — implement cycle detection; compute
and display the topological ordering for both a valid DAG and a graph with a cycle;
show what Kahn's algorithm looks like step by step.

**3 · Agent Tool Call DAG** — model a multi-step AI agent workflow as a DAG,
visualise the tool call graph, run topological sort to get the execution order,
and inject a cycle to show what a looping agent workflow looks like.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import deque

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Software Dependency Digraph

We model a Python project's import dependencies as a directed graph — each arc
$(u, v)$ means "module $u$ imports module $v$." We compute in/out-degrees,
identify sources (modules with no dependencies), sinks (modules nothing depends on),
and check for circular imports.


In [ ]:
# ── 1 · Software Dependency Digraph ──────────────────────────────────────────

D = nx.DiGraph()

# Modules: (module, type)
modules = {
    'main':       'entry',
    'auth':       'service',
    'api':        'service',
    'db':         'service',
    'cache':      'service',
    'utils':      'util',
    'config':     'util',
    'logging':    'util',
    'crypto':     'util',
    'validators': 'util',
}
D.add_nodes_from(modules.keys())

# Dependencies: (importer, imported)
deps = [
    ('main',       'auth'),
    ('main',       'api'),
    ('main',       'config'),
    ('auth',       'db'),
    ('auth',       'crypto'),
    ('auth',       'logging'),
    ('api',        'db'),
    ('api',        'cache'),
    ('api',        'validators'),
    ('api',        'logging'),
    ('db',         'config'),
    ('db',         'logging'),
    ('cache',      'config'),
    ('validators', 'utils'),
    ('crypto',     'utils'),
]
D.add_edges_from(deps)

# ── Degree analysis ────────────────────────────────────────────────────────────
print(f"Dependency Digraph: |V|={D.number_of_nodes()}, |E|={D.number_of_edges()}")
print(f"Is DAG: {nx.is_directed_acyclic_graph(D)}")
print()
print(f"{'Module':<14} {'out+':>5} {'in-':>4}  {'Role'}")
print("─" * 50)
for mod in sorted(D.nodes()):
    out = D.out_degree(mod)
    inn = D.in_degree(mod)
    role = ('SOURCE — no deps' if inn == 0
            else 'SINK — nothing imports this' if out == 0
            else '')
    print(f"  {mod:<12}  {out:>3}+  {inn:>3}-  {role}")

sources = [v for v in D.nodes() if D.in_degree(v) == 0]
sinks   = [v for v in D.nodes() if D.out_degree(v) == 0]
print(f"\nSources (no deps):        {sorted(sources)}")
print(f"Sinks   (nothing imports): {sorted(sinks)}")

# ── Visualise ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
pos = nx.spring_layout(D, seed=7, k=2.5)

type_colors = {'entry': TS_RED, 'service': TS_BLUE,
               'util': TS_GREEN}
node_colors = [type_colors[modules[v]] for v in D.nodes()]

nx.draw_networkx_edges(D, pos, ax=ax, edge_color=TS_GRAY, alpha=0.6,
                       arrows=True, arrowsize=18, width=1.8,
                       connectionstyle='arc3,rad=0.05')
nx.draw_networkx_nodes(D, pos, ax=ax, node_color=node_colors,
                       node_size=900, alpha=0.92)
nx.draw_networkx_labels(D, pos, ax=ax, font_size=8,
                        font_color='white', font_weight='bold')

legend = [mpatches.Patch(color=TS_RED,   label='Entry point'),
          mpatches.Patch(color=TS_BLUE,  label='Service'),
          mpatches.Patch(color=TS_GREEN, label='Utility (sink — high reuse)')]
ax.legend(handles=legend, loc='upper left', fontsize=9)
ax.set_title('Software Dependency Digraph\nArrows point from importer → imported',
             pad=12, fontsize=11)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The degree table reveals the structure immediately:

- **Sources** (in-degree = 0): `main` — nothing imports it; it is the entry point.
- **Sinks** (out-degree = 0): `utils`, `config`, `logging` — pure utilities imported
  by many but importing nothing. These are the high-reuse, high-criticality modules.
  A vulnerability in `utils` or `logging` propagates to every module that imports them.

The is-DAG check confirms there are no circular imports. Introducing a cycle —
say, making `utils` import `api` — would be flagged immediately.

**The security principle:** the modules with high in-degree (many importers) and
low out-degree (few dependencies) are the libraries most worth auditing. They are
sinks in the dependency graph — everything flows through them, and they themselves
are clean of external dependencies, so any vulnerability is self-contained to that
module. The `logging` module with high in-degree is a particularly interesting
target: if an attacker can influence log output (log injection), they reach every
module that calls the logger.


### 2 · DAG Detection and Topological Sort — Kahn's Algorithm

We implement Kahn's algorithm from scratch to make the topological ordering process
transparent, then run it on both a clean DAG and a graph with a planted cycle.


In [ ]:
# ── 2 · Kahn's Algorithm — Topological Sort ──────────────────────────────────

def kahns_topological_sort(G):
    """
    Kahn's algorithm for topological ordering.
    Returns (order, cycle_nodes) where cycle_nodes is non-empty iff a cycle exists.
    """
    in_deg  = {v: G.in_degree(v) for v in G.nodes()}
    queue   = deque([v for v in G.nodes() if in_deg[v] == 0])
    order   = []
    steps   = []   # record each step for display

    while queue:
        v = queue.popleft()
        order.append(v)
        steps.append((v, sorted(queue), dict(in_deg)))
        for u in G.successors(v):
            in_deg[u] -= 1
            if in_deg[u] == 0:
                queue.append(u)

    # If order doesn't include all vertices, there's a cycle
    cycle_nodes = [v for v in G.nodes() if v not in order]
    return order, cycle_nodes

# ── Apply to clean DAG ────────────────────────────────────────────────────────
order, cycle = kahns_topological_sort(D)
print("Clean DAG — topological order (install/import sequence):")
for i, v in enumerate(order):
    print(f"  Step {i+1:>2}: {v}")
print(f"\nCycle detected: {bool(cycle)}  ({'nodes: ' + str(cycle) if cycle else 'none — valid DAG ✓'})")

# ── Plant a circular dependency ───────────────────────────────────────────────
D_cyclic = D.copy()
D_cyclic.add_edge('utils', 'api')   # utils → api → ... → utils (cycle!)

order_c, cycle_c = kahns_topological_sort(D_cyclic)
print(f"\nWith circular dependency utils→api→validators→utils:")
print(f"  Topological order (partial): {order_c}")
print(f"  Cycle detected: {bool(cycle_c)}")
print(f"  Nodes stuck in cycle: {sorted(cycle_c)}")

# ── Visualise topological levels ──────────────────────────────────────────────
# Assign a 'level' to each node = longest path from any source
levels = {}
for v in order:
    preds = list(D.predecessors(v))
    levels[v] = 0 if not preds else max(levels[p] for p in preds) + 1

fig, ax = plt.subplots(figsize=(13, 5))
max_level = max(levels.values())
level_groups = {}
for v, lv in levels.items():
    level_groups.setdefault(lv, []).append(v)

pos_topo = {}
for lv, nodes in level_groups.items():
    for i, v in enumerate(sorted(nodes)):
        pos_topo[v] = (lv, -(i - len(nodes)/2))

nx.draw_networkx_edges(D, pos_topo, ax=ax, edge_color=TS_GRAY, alpha=0.5,
                       arrows=True, arrowsize=20, width=2,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_nodes(D, pos_topo, ax=ax,
                       node_color=[type_colors[modules[v]] for v in D.nodes()],
                       node_size=900, alpha=0.92)
nx.draw_networkx_labels(D, pos_topo, ax=ax, font_size=8,
                        font_color='white', font_weight='bold')

# Level annotations
for lv in range(max_level + 1):
    ax.text(lv, max(-(i - len(level_groups.get(lv,[])) / 2)
                    for i in range(len(level_groups.get(lv,[])))) + 1.2,
            f'Level {lv}', ha='center', fontsize=8, color=TS_GRAY, style='italic')
    ax.axvline(lv, color='#eeeeee', lw=1, alpha=0.8, zorder=0)

ax.set_title('Topological Ordering — Nodes Arranged by Level (Kahn\'s Algorithm)\n'
             'Arrows always point left → right   |   Each level can execute in parallel',
             pad=12, fontsize=11)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The topological ordering lays out the modules so every arrow points left-to-right —
the cleanest visualisation of "what must come before what." All modules at the
same level can be executed (or compiled, or installed) in parallel without
dependency conflicts.

When the circular dependency `utils → api` is added, Kahn's algorithm stalls.
It correctly reports the partial order for everything not involved in the cycle,
and names the nodes stuck in the cycle. This is exactly how `pip` and other
package managers detect circular dependencies — the same algorithm.

**The security insight:** a circular dependency is not just a packaging inconvenience.
It means two components are mutually dependent — a vulnerability in either propagates
to both, and patching one without the other is incoherent. Circular dependencies
in security-critical code (authentication libraries, crypto modules) are red flags
worth escalating immediately.


### 3 · Agent Tool Call DAG

We model a multi-step AI agent workflow as a DAG. Each node is a tool call or
decision point; each arc is a data dependency (the output of one tool feeds into
another). We visualise the execution graph, run topological sort to get the correct
execution order, and then inject a cycle to demonstrate what a looping agent
workflow looks like — a real failure mode in agentic AI systems.


In [ ]:
# ── 3 · Agent Tool Call DAG ──────────────────────────────────────────────────

# An AI agent performing a security audit workflow:
# 1. Receives a task (source)
# 2. Calls tools to gather data
# 3. Synthesises findings
# 4. Produces a report (sink)

agent_dag = nx.DiGraph()

# Tool calls as nodes
tools = {
    'task_input':       'input',
    'fetch_user_list':  'data',
    'fetch_audit_log':  'data',
    'check_mfa_status': 'analysis',
    'check_suspensions':'analysis',
    'analyse_log_gaps': 'analysis',
    'synthesise_findings': 'synthesis',
    'format_report':    'output',
    'send_report':      'output',
}
agent_dag.add_nodes_from(tools.keys())

# Data dependencies: arc = "output of source feeds input of target"
tool_deps = [
    ('task_input',       'fetch_user_list'),
    ('task_input',       'fetch_audit_log'),
    ('fetch_user_list',  'check_mfa_status'),
    ('fetch_user_list',  'check_suspensions'),
    ('fetch_audit_log',  'analyse_log_gaps'),
    ('check_mfa_status', 'synthesise_findings'),
    ('check_suspensions','synthesise_findings'),
    ('analyse_log_gaps', 'synthesise_findings'),
    ('synthesise_findings', 'format_report'),
    ('format_report',    'send_report'),
]
agent_dag.add_edges_from(tool_deps)

topo_order, cycle_nodes = kahns_topological_sort(agent_dag)
print("Agent Workflow — Tool Execution Order (topological sort):")
for i, tool in enumerate(topo_order):
    role = tools[tool]
    print(f"  {i+1:>2}. [{role:>10}]  {tool}")
print(f"\nCycle detected: {bool(cycle_nodes)}  → workflow terminates safely ✓")

# ── Inject a cycle: format_report feeds back to fetch_user_list ───────────────
agent_dag_cyclic = agent_dag.copy()
agent_dag_cyclic.add_edge('format_report', 'fetch_user_list')

_, cycle_c = kahns_topological_sort(agent_dag_cyclic)
print(f"\nWith injected loop (format_report → fetch_user_list):")
print(f"  Cycle detected: {bool(cycle_c)}")
print(f"  Stuck nodes: {sorted(cycle_c)}")
print(f"  → Agent would loop indefinitely refetching and reformatting ⚠")

# ── Visualise both ────────────────────────────────────────────────────────────
type_pal = {'input': TS_GREEN, 'data': TS_BLUE, 'analysis': TS_AMBER,
            'synthesis': '#8e44ad', 'output': TS_RED}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, graph, title in [
    (axes[0], agent_dag,        'Valid Agent Workflow DAG
(terminates correctly)'),
    (axes[1], agent_dag_cyclic, 'Cyclic Agent Workflow
(infinite loop ⚠)'),
]:
    pos = nx.spring_layout(graph, seed=11, k=3.0)
    colors = [type_pal[tools[v]] for v in graph.nodes()]

    # Highlight cycle edges in red
    cycle_edges = [(u,v) for u,v in graph.edges()
                   if u not in agent_dag.edges() or v not in
                   [w for _,w in agent_dag.edges()]]
    normal_edges = [(u,v) for u,v in graph.edges()
                    if (u,v) not in cycle_edges]

    nx.draw_networkx_edges(graph, pos, ax=ax, edgelist=normal_edges,
                           edge_color=TS_GRAY, alpha=0.6, arrows=True,
                           arrowsize=18, width=2,
                           connectionstyle='arc3,rad=0.08')
    if cycle_edges:
        nx.draw_networkx_edges(graph, pos, ax=ax, edgelist=cycle_edges,
                               edge_color=TS_RED, alpha=0.9, arrows=True,
                               arrowsize=22, width=3,
                               connectionstyle='arc3,rad=0.2')
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=colors,
                           node_size=800, alpha=0.92)
    nx.draw_networkx_labels(graph, pos, ax=ax, font_size=7,
                            font_color='white', font_weight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    ax.set_facecolor('white')

# Shared legend
legend_elements = [
    mpatches.Patch(color=TS_GREEN,  label='Input'),
    mpatches.Patch(color=TS_BLUE,   label='Data fetch'),
    mpatches.Patch(color=TS_AMBER,  label='Analysis'),
    mpatches.Patch(color='#8e44ad', label='Synthesis'),
    mpatches.Patch(color=TS_RED,    label='Output / cycle edge'),
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.patch.set_facecolor('white')
plt.suptitle('Agent Tool Call DAG — Valid vs Cyclic Workflow',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**What do you see?**

The valid workflow (left) is a clean DAG — the topological order gives the
correct execution sequence: input first, parallel data fetches, parallel analyses,
then synthesis and output. No tool starts before its inputs are ready.

The cyclic workflow (right) has a red arc from `format_report` back to
`fetch_user_list`. The topological sort stalls — those nodes are stuck because
their in-degrees never reach zero. In a real agent system, this would mean:

1. The agent fetches user data and generates a report.
2. The report's output triggers another user data fetch.
3. The new user data triggers another report.
4. Repeat indefinitely — infinite loop.

This is a real safety property of agentic AI systems: **the tool call graph must
be acyclic** for the workflow to terminate. Detecting cycles in the tool call graph
before execution begins is a concrete security check that Agent Scrutiny-style
frameworks implement. The mathematics behind that check is exactly what you just
built: DAG detection via Kahn's algorithm.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. STRONGLY CONNECTED COMPONENTS
#    NetworkX has nx.strongly_connected_components(G).
#    Add a second back-edge to D_cyclic to create a larger SCC.
#    Compute the SCCs and draw a condensation graph — a DAG where each
#    node is an SCC. What does the condensation always look like, and why?
#
# 2. LONGEST PATH IN A DAG
#    Implement longest_path(G) for a DAG using topological sort:
#      - Process vertices in topological order
#      - For each vertex v, dist[v] = max(dist[u] + 1 for all predecessors u)
#    Apply to the dependency digraph D. What is the longest dependency chain?
#    Why does the longest path matter for build time and for attack chains?
#
# 3. AGENT PERMISSION PROPAGATION
#    Model a more complex agent system where each tool call has a required
#    permission level (1=low, 3=high):
#      fetch_user_list: 2, fetch_audit_log: 2, check_mfa_status: 1, ...
#    The agent starts with permission level 1.
#    A tool call is only safe if the agent's current level >= required level.
#    Simulate propagating permission requirements through the DAG topologically.
#    Which tool calls require privilege escalation? Is escalation always safe?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Directed graph** | $(V, E)$ with ordered arcs | Dependencies, data flow, attack direction |
| **In-degree $\deg^-$** | Arcs entering $v$ | How many things depend on $v$ |
| **Out-degree $\deg^+$** | Arcs leaving $v$ | How many things $v$ depends on |
| **Source** | $\deg^- = 0$ | Entry point; no-dependency module |
| **Sink** | $\deg^+ = 0$ | Target; leaf utility |
| **Reachability $\rightsquigarrow$** | Directed path exists | Attacker can move from entry to target |
| **DAG** | No directed cycles | Computable; terminating; safe workflow |
| **Topological order** | All arcs point forward | Execution order respecting all dependencies |
| **Cycle = no topo order** | Kahn's algorithm stalls | Circular dependency; looping agent; unresolvable build |

---

## Up Next

**Module 04 · Unit 03 — Neural Network Topology**

We apply the full graph and DAG toolkit to the structures that power modern AI.
Transformer attention is a weighted bipartite graph. The autograd computation
graph is a DAG. An MCP agent tool call network is a directed graph with a
specific DAG requirement. We build and visualise all three, connect them to
the formal definitions, and analyse what "removing a node" means in each context.

$\rightarrow$ `modules/module-04/unit-03-neural-network-topology.ipynb`
